In [ ]:
# Run first on Google Colab
!pip install qutip -q

# §2 Quantum Fourier Transform

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Implement the QFT from the product formula and verify unitarity
- Relate the QFT circuit to the classical Cooley–Tukey FFT
- Analyse approximate QFT truncation error vs gate count


In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

print(f"QuTiP {qt.__version__}")


In [ ]:
# ── Standard single-qubit states and gates (used throughout §2) ─────────────
zero  = qt.basis(2, 0)           # |0⟩
one   = qt.basis(2, 1)           # |1⟩
plus  = (zero + one).unit()      # |+⟩
minus = (zero - one).unit()      # |−⟩

X  = qt.sigmax()
Y  = qt.sigmay()
Z  = qt.sigmaz()
I  = qt.qeye(2)
P0 = zero * zero.dag()           # |0⟩⟨0|
P1 = one  * one.dag()            # |1⟩⟨1|


---
## Part 2: Quantum Fourier Transform

### 2.1 Definition and unitarity

The **$n$-qubit QFT** is the DFT matrix acting on amplitudes:
$$\mathsf{QFT}_N|k\rangle = \frac{1}{\sqrt{N}}\sum_{j=0}^{N-1}\omega_N^{jk}|j\rangle, \qquad \omega_N = e^{2\pi i/N}, \quad N = 2^n$$

As a matrix: $(\mathsf{QFT}_N)_{jk} = \omega_N^{jk}/\sqrt{N}$.

**Unitarity:** $\mathsf{QFT}_N^\dagger\mathsf{QFT}_N = \mathbb{I}$ (columns are orthonormal — orthogonality of complex exponentials).


In [ ]:
def qft_matrix(n):
    """n-qubit QFT as a QuTiP Qobj (DFT matrix on 2^n-dim space)."""
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
    return qt.Qobj(F, dims=[[2]*n, [2]*n])

F2 = qft_matrix(2)   # 2-qubit QFT (4×4)
F3 = qft_matrix(3)   # 3-qubit QFT (8×8)

print("QFT_4 matrix (real parts):")
print(np.round(F2.full().real, 3))

# Unitarity
print("\nQFT†·QFT = I (n=2):", np.allclose((F2.dag()*F2).full(), np.eye(4)))
print("QFT†·QFT = I (n=3):", np.allclose((F3.dag()*F3).full(), np.eye(8)))

# Verify QFT_4|k> = (1/2) sum_j i^{jk} |j>  (since ω_4 = i)
ket1 = qt.basis(4, 1)
ket1.dims = [[2,2],[1,1]]
out = F2 * ket1
expected = np.array([1, 1j, -1, -1j]) / 2.0   # j=0..3, k=1: omega^j = i^j
print("\nQFT_4|1> =", np.round(out.full().flatten(), 4))
print("expected  =", np.round(expected, 4))


### 2.2 The QFT circuit (product formula)

**Theorem (Product formula):**
$$\mathsf{QFT}_N|k_1 k_2\cdots k_n\rangle = \bigotimes_{l=1}^{n}\frac{1}{\sqrt{2}}\bigl(|0\rangle + e^{2\pi i\,[0.k_{n-l+1}\cdots k_n]}|1\rangle\bigr)$$

The output is a **tensor product** of single-qubit states — enabling an efficient qubit-by-qubit circuit.
Each qubit $l$ of the output depends only on the last $l$ bits of the input.

**Circuit:** For qubit $i$ (0-indexed):
1. Apply $\mathsf{H}$
2. For each later qubit $j > i$: apply controlled-$\mathsf{R}_{j-i+1}$ (control = $j$, target = $i$)

Then reverse qubit order with $\lfloor n/2\rfloor$ SWAPs.
Total gates: $\frac{n(n+1)}{2} + \lfloor n/2\rfloor = O(n^2)$.

Where $\mathsf{R}_m = \mathrm{diag}(1, e^{2\pi i/2^m})$ adds a phase of $e^{2\pi i/2^m}$ to $|1\rangle$.


In [ ]:
# ── Helper: apply single-qubit gate to qubit i in n-qubit system ──────────────
def apply_to_qubit(gate, n, qubit):
    ops = [qt.qeye(2)] * n
    ops[qubit] = gate
    return qt.tensor(ops)

# ── Helper: controlled-U (control=ctrl, target=tgt) in n-qubit system ─────────
def controlled_u(U, n, ctrl, tgt):
    P0 = qt.basis(2,0) * qt.basis(2,0).dag()
    P1 = qt.basis(2,1) * qt.basis(2,1).dag()
    ops0 = [qt.qeye(2)] * n;  ops0[ctrl] = P0
    ops1 = [qt.qeye(2)] * n;  ops1[ctrl] = P1;  ops1[tgt] = U
    return qt.tensor(ops0) + qt.tensor(ops1)

# ── Helper: SWAP gate between qubits i and j ──────────────────────────────────
def swap_gate(n, i, j):
    N = 2**n
    mat = np.zeros((N, N), dtype=complex)
    for k in range(N):
        bits = list(format(k, f'0{n}b'))
        bits[i], bits[j] = bits[j], bits[i]
        mat[int(''.join(bits), 2), k] = 1.0
    return qt.Qobj(mat, dims=[[2]*n, [2]*n])

# ── QFT circuit ───────────────────────────────────────────────────────────────
def qft_circuit(n):
    """Build QFT_N from the product formula circuit."""
    dims = [[2]*n, [2]*n]
    U = qt.tensor([qt.qeye(2)] * n)
    H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1, n, i) * U
        for j in range(i+1, n):
            m = j - i + 1
            Rm = qt.Qobj(np.diag([1.0, np.exp(2j*np.pi/2**m)]))
            U = controlled_u(Rm, n, j, i) * U
    for i in range(n // 2):
        U = swap_gate(n, i, n-1-i) * U
    return U

print("Building QFT circuits...")
Fc2 = qft_circuit(2)
Fc3 = qft_circuit(3)
print("Done.")


In [ ]:
# Verify circuit == exact DFT matrix
print("Circuit = DFT matrix (n=2):", np.allclose(Fc2.full(), F2.full()))
print("Circuit = DFT matrix (n=3):", np.allclose(Fc3.full(), F3.full()))

# Trace through n=3 circuit for |k> = |101> (k=5)
k_val = 5
ket_k = qt.basis(8, k_val);  ket_k.dims = [[2,2,2],[1,1,1]]
out_circ = Fc3 * ket_k
out_dft  = F3  * ket_k

print(f"\nQFT|{k_val}⟩ via circuit and matrix agree:", np.allclose(out_circ.full(), out_dft.full()))

# Visualise |QFT|k>|^2 for all k
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for k_test in range(4):
    ket = qt.basis(8, k_test);  ket.dims = [[2,2,2],[1,1,1]]
    probs = np.abs((F3 * ket).full().flatten())**2
    axes[k_test].bar(range(8), probs)
    axes[k_test].set_title(f"|QFT|{k_test}⟩|²")
    axes[k_test].set_xlabel("basis state j")
    axes[k_test].set_xticks(range(8))
plt.tight_layout()
plt.show()


### 2.3 QFT diagonalises the cyclic shift

The **cyclic shift** operator $\hat{X}|k\rangle = |k+1 \bmod N\rangle$ is the quantum analogue of the derivative operator (or circulant shift).

The QFT diagonalises $\hat{X}$:
$$\mathsf{QFT}_N^\dagger\,\hat{X}\,\mathsf{QFT}_N = \hat{Z}^{-1},$$
where $\hat{Z}|k\rangle = \omega_N^k|k\rangle$ is multiplication by $\omega_N^k$ in the Fourier basis.

The eigenvectors of $\hat{X}$ are the QFT basis states $\mathsf{QFT}_N^\dagger|j\rangle$, with eigenvalues $\omega_N^j$.


In [ ]:
n = 3;  N = 2**n

# Build cyclic shift X_hat: |k> -> |(k+1) mod N>
shift_mat = np.zeros((N, N))
for k in range(N):
    shift_mat[(k+1) % N, k] = 1.0
Xhat = qt.Qobj(shift_mat, dims=[[2]*n, [2]*n])

# Build diagonal phase operator Z_hat: |k> -> omega^k |k>
omega = np.exp(2j * np.pi / N)
Zhat = qt.Qobj(np.diag([omega**k for k in range(N)]), dims=[[2]*n, [2]*n])

# Verify QFT† X_hat QFT = Z_hat^{-1}
lhs = F3.dag() * Xhat * F3
rhs = Zhat.dag()   # Z_hat^{-1} = Z_hat†
print("QFT† X̂ QFT = Ẑ⁻¹:", np.allclose(lhs.full(), rhs.full()))

# Verify eigenvectors of X_hat are QFT columns
evals, evecs = Xhat.eigenstates()
print("\nEigenvalues of cyclic shift (should be ω^j for j=0..7):")
print(np.round(sorted(evals), 3))
expected_evals = sorted([omega**j for j in range(N)])
print("Expected:", np.round(expected_evals, 3))


### Exercise 2.3 — QFT₄ by hand

**(a)** Apply $\mathsf{QFT}_4$ to $|1\rangle$ and $|2\rangle$ using the product formula.
Express each result as a tensor product of single-qubit states $\frac{1}{\sqrt{2}}(|0\rangle + e^{i\phi}|1\rangle)$ for appropriate $\phi$.

**(b)** Verify using the matrix definition that $\mathsf{QFT}_4|k\rangle = \frac{1}{2}\sum_{j=0}^{3}i^{jk}|j\rangle$.


In [ ]:
# YOUR CODE HERE

# (a) Apply QFT_4 to |1> and |2> using the product formula
# For |k> = |k1 k2>, the product formula gives:
#   QFT_4|k> = (1/sqrt(2))(|0>+e^{2πi[0.k2]}|1>) ⊗ (1/sqrt(2))(|0>+e^{2πi[0.k1k2]}|1>)
# Then swap the two output qubits.

# ket1_manual = ...   (build manually from product formula for k=1=[01]_2)
# ket2_manual = ...   (for k=2=[10]_2)

# (b) Verify with matrix
F_4 = qft_matrix(2)
# ket1 = qt.basis(4,1); ket1.dims = [[2,2],[1,1]]
# out = F_4 * ket1
# expected = ...


In [ ]:
#@title Solution — Exercise 2.3  {display-mode: "form"}
F_4 = qft_matrix(2)

# ── (a) QFT_4|1> by product formula ─────────────────────────────────────────
# k=1 = [k1 k2] = [0 1] in binary
# Factor l=2 (most significant output qubit): 1/√2(|0>+e^{2πi[0.k2]}|1>) = 1/√2(|0>+e^{πi}|1>) = 1/√2(|0>-|1>)
# Factor l=1 (least significant output qubit after swap): 1/√2(|0>+e^{2πi[0.k1k2]}|1>) = 1/√2(|0>+e^{πi/2}|1>)
# But product formula gives output in REVERSED order before the SWAP, so after SWAP:
# qubit 1 <- factor with most phase (depends on all bits)
# Actually: factor for l=1 is 1/√2(|0>+e^{2πi[0.k2]}|1>) = 1/√2(|0>+e^{πi}|1>) = minus
#           factor for l=2 is 1/√2(|0>+e^{2πi[0.k1k2]}|1>) = 1/√2(|0>+e^{πi/2}|1>) = 1/√2(|0>+i|1>)
# After SWAP: qubit1=factor_l2, qubit2=factor_l1

q1_k1 = (zero + np.exp(1j*np.pi/2)*one).unit()   # 1/√2(|0>+i|1>)
q2_k1 = (zero + np.exp(1j*np.pi)*one).unit()      # 1/√2(|0>-|1>) = |->

ket1_manual = qt.tensor(q1_k1, q2_k1)
ket1 = qt.basis(4,1); ket1.dims = [[2,2],[1,1]]
out1 = F_4 * ket1
print("QFT_4|1> manual:", np.round(ket1_manual.full().flatten(), 4))
print("QFT_4|1> matrix:", np.round(out1.full().flatten(), 4))
print("Match:", np.allclose(ket1_manual.full(), out1.full()))

# ── (b) Verify QFT_4|k> = (1/2) sum_j i^{jk} |j> ────────────────────────────
print()
for k in range(4):
    ket_k = qt.basis(4,k); ket_k.dims = [[2,2],[1,1]]
    out = F_4 * ket_k
    exp = np.array([1j**(j*k) for j in range(4)]) / 2.0
    print(f"k={k}: match = {np.allclose(out.full().flatten(), exp)}")


### 2.4 Approximate QFT

**Definition:** Drop all controlled-$\mathsf{R}_m$ gates with $m > m_{\max}$.
Uses $O(n\, m_{\max})$ gates instead of $O(n^2)$.

**Theorem (error bound):**
$$\|\mathsf{QFT} - \mathsf{QFT}_{m_{\max}}\|_F \leq \frac{2\pi n}{2^{m_{\max}}}$$

Choosing $m_{\max} = \lceil\log_2(n/\varepsilon)\rceil$ gives Frobenius error $< \varepsilon$ with $O(n\log(n/\varepsilon))$ gates — matching the complexity of a sparse DFT.


In [ ]:
def qft_circuit_approx(n, m_max):
    """Approximate QFT: drop controlled-R_m gates with m > m_max."""
    dims = [[2]*n, [2]*n]
    U = qt.tensor([qt.qeye(2)] * n)
    H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1, n, i) * U
        for j in range(i+1, n):
            m = j - i + 1
            if m <= m_max:                   # keep only rotations up to m_max
                Rm = qt.Qobj(np.diag([1.0, np.exp(2j*np.pi/2**m)]))
                U = controlled_u(Rm, n, j, i) * U
    for i in range(n // 2):
        U = swap_gate(n, i, n-1-i) * U
    return U

# ── Compute Frobenius error vs m_max for several n ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for n_test in [3, 4, 5]:
    N_test = 2**n_test
    F_exact = qft_matrix(n_test)
    m_vals = range(1, n_test+1)
    errors = []
    for m in m_vals:
        F_approx = qft_circuit_approx(n_test, m)
        diff = (F_exact - F_approx).full()
        err = np.linalg.norm(diff, 'fro')
        errors.append(err)
    ax1.semilogy(list(m_vals), errors, 'o-', label=f'n={n_test}')
    # Plot theoretical bound
    bounds = [2*np.pi*n_test/2**m for m in m_vals]
    ax1.semilogy(list(m_vals), bounds, '--', alpha=0.5)

ax1.set_xlabel('m_max (cutoff)')
ax1.set_ylabel('Frobenius error')
ax1.set_title('Approx QFT error vs cutoff\n(solid=actual, dashed=bound)')
ax1.legend()

# Gate count comparison
n_range = range(2, 8)
for m_max in [2, 3, 4]:
    counts = [n * m_max for n in n_range]
    ax2.plot(list(n_range), counts, label=f'm_max={m_max}')
exact_counts = [n*(n+1)//2 + n//2 for n in n_range]
ax2.plot(list(n_range), exact_counts, 'k--', label='exact O(n²)')
ax2.set_xlabel('n (qubits)')
ax2.set_ylabel('gate count')
ax2.set_title('Approximate vs exact QFT gate count')
ax2.legend()
plt.tight_layout()
plt.show()


### Exercise 2.4 — Finding the minimum cutoff for a target error

The error bound guarantees $\|\mathsf{QFT} - \mathsf{QFT}_{m_{\max}}\|_F \leq \dfrac{2\pi n}{2^{m_{\max}}}$.

**(a)** Given a target Frobenius error $\varepsilon$ and qubit count $n$, derive the smallest integer $m_{\max}$ that satisfies the bound analytically.

**(b)** For $n = 5$ and $\varepsilon \in \{0.5, 0.1, 0.01\}$, compute the analytical minimum $m_{\max}$ from part (a).
Then measure the *actual* Frobenius error `qft_circuit_approx(5, m_max)` for each case and verify it is below $\varepsilon$.

**(c)** Is the analytical bound tight? For each $\varepsilon$, check whether $m_{\max} - 1$ also achieves error $< \varepsilon$ in practice.

In [ ]:
# YOUR CODE HERE

import math

n = 5
epsilons = [0.5, 0.1, 0.01]

# (a) Derive m_max_min analytically:
# 2*pi*n / 2**m_max <= eps  =>  2**m_max >= 2*pi*n/eps  =>  m_max >= log2(2*pi*n/eps)
# m_max_min = ...

# (b) For each epsilon, compute m_max_min, then measure actual error
# ...

# (c) Check whether m_max_min - 1 also works in practice
# ...

In [ ]:
#@title Solution — Exercise 2.4  {display-mode: "form"}
import math

n = 5
F_exact = qft_matrix(n)
epsilons = [0.5, 0.1, 0.01]

print(f"{'eps':>6}  {'m_min (bound)':>14}  {'actual err @ m_min':>20}  {'actual err @ m_min-1':>22}  {'bound tight?':>12}")
print('-' * 82)
for eps in epsilons:
    # (a) smallest m_max satisfying 2*pi*n / 2**m <= eps
    m_min = math.ceil(math.log2(2 * math.pi * n / eps))

    # (b) actual error at m_min
    err_m = np.linalg.norm((F_exact - qft_circuit_approx(n, m_min)).full(), 'fro')

    # (c) actual error at m_min - 1
    err_m1 = np.linalg.norm((F_exact - qft_circuit_approx(n, m_min - 1)).full(), 'fro')
    tight = err_m1 >= eps

    print(f"{eps:>6.3f}  {m_min:>14d}  {err_m:>20.6f}  {err_m1:>22.6f}  {'yes' if tight else 'no':>12}")

---
## Summary

| Topic | Key result |
|-------|-----------|
| QFT definition | $\mathsf{QFT}_N|k\rangle = \frac{1}{\sqrt{N}}\sum_{j=0}^{N-1}\omega_N^{jk}|j\rangle$; unitary (DFT on amplitudes) |
| QFT circuit | $O(n^2)$ gates via product formula; controlled-$\mathsf{R}_m$ gates + bit-reversal SWAPs |
| Cyclic shift | $\mathsf{QFT}^\dagger\,\hat{X}\,\mathsf{QFT} = \hat{Z}^{-1}$; QFT diagonalises the cyclic shift operator |
| Approximate QFT | Drop gates with $m > m_{\max}$; Frobenius error $\leq \frac{2\pi n}{2^{m_{\max}}}$; $O(n\log(n/\varepsilon))$ gates |

**Next:** Notebook 3 applies the QFT as a subroutine in Quantum Phase Estimation (§3).